# Real NASA data: cleaning, exclusions and readiness
This notebook does not fit a model or manufacture negative labels. It calls the reusable preparation code and publishes full pipeline statecharts.

In [1]:
from pathlib import Path
import json
import pandas as pd
from firecast.real import prepare_real, require_ready
from firecast.statecharts import write_statecharts
root = Path.cwd() if (Path.cwd() / 'configs').exists() else Path.cwd().parent
report=prepare_real(root/'data/nasa-lebanon',root/'configs/real-preparation.json',root/'outputs/real-preparation')
write_statecharts(report,root/'docs/statecharts.md')

In [2]:
pd.DataFrame({p:r['labels'] for p,r in report['policies'].items()}).fillna(0).astype(int)

,conservative,strict
uncertain,1558416,1557712
positive,6468,3883
excluded,0,3289


In [3]:
audit=pd.read_csv(root/'outputs/real-preparation/detection-audit.csv.gz')
audit.loc[audit.flag_level.ne('unflagged'), ['detected_at','product','confidence','type','flag_level','flag_reason','conservative_action','strict_action']].head(10)

,detected_at,product,confidence,type,flag_level,flag_reason,conservative_action,strict_action
38,2002-08-08 11:35:00+00:00,modis,0,0,uncertain,low sensor-specific confidence,retain_detection_candidate,quarantine_entire_cell_window
39,2002-08-08 11:35:00+00:00,modis,0,0,uncertain,low sensor-specific confidence,retain_detection_candidate,quarantine_entire_cell_window
120,2003-06-26 08:09:00+00:00,modis,26,0,uncertain,low sensor-specific confidence,retain_detection_candidate,quarantine_entire_cell_window
132,2003-08-27 08:21:00+00:00,modis,14,0,uncertain,low sensor-specific confidence,retain_detection_candidate,quarantine_entire_cell_window
133,2003-08-28 10:37:00+00:00,modis,15,0,uncertain,low sensor-specific confidence,retain_detection_candidate,quarantine_entire_cell_window
166,2003-11-02 10:25:00+00:00,modis,24,0,uncertain,low sensor-specific confidence,retain_detection_candidate,quarantine_entire_cell_window
196,2004-09-12 10:09:00+00:00,modis,0,0,uncertain,low sensor-specific confidence,retain_detection_candidate,quarantine_entire_cell_window
197,2004-09-12 10:09:00+00:00,modis,0,0,uncertain,low sensor-specific confidence,retain_detection_candidate,quarantine_entire_cell_window
219,2004-10-14 23:49:00+00:00,modis,20,0,uncertain,low sensor-specific confidence,retain_detection_candidate,quarantine_entire_cell_window
264,2005-09-07 10:56:00+00:00,modis,17,0,uncertain,low sensor-specific confidence,retain_detection_candidate,quarantine_entire_cell_window


In [4]:
try:
    require_ready(report)
except ValueError as exc:
    print(str(exc))
print('No real-data model was fitted.')

Real-data training blocked: no compatible satellite observability masks: no verified negative labels; candidate bbox lacks a verified national/vegetation mask; historically available vegetation/terrain features missing; POWER history has no verified issue/availability timestamps for operational evaluation; complete multi-year forecast meteorology not acquired
No real-data model was fitted.


Read `docs/statecharts.md` for eight Mermaid diagrams covering the whole pipeline, cleaning/exclusion decisions, feature timing, seasonal/logistic models, validation/test separation and inference. NASA environmental product metadata are in `data/earthdata-plan/`; they are not downloaded rasters. The candidate rectangle is not a verified national forecast grid.